# Baseline vs poda estructurada (cadena) vs ResNet-18 podado (IMP)

Tres puntos de comparación sobre **CIFAR-10 test** (mismas métricas de cómputo que `baseline_vs_pruned.ipynb`):

| Modelo | Origen típico | Nota |
|--------|-----------------|------|
| **ResNet baseline** | `round_00/model_state.pt` de un run IMP | Red densa al acabar la ronda 0 (misma familia que el IMP podado). |
| **Structured (cadena)** | `model_final.pt` de un run `structured_ticket` | `NarrowableChainCnn`: **menos parámetros de verdad** (ancho recortado); arquitectura distinta a ResNet. |
| **ResNet IMP final** | `model_final.pt` del mismo run IMP | Misma forma que el baseline; dispersidad alta (mucho cero); tamaño en RAM/disco igual al baseline. |

Ajusta las rutas en la siguiente celda a tus carpetas bajo `runs/`.

In [ ]:
"""Rutas del repo, CIFAR-10 y checkpoints de los tres experimentos."""

from __future__ import annotations

import sys
from pathlib import Path


def resolve_repo_root() -> Path:
    """Sube desde notebooks/ hasta la raíz (directorio con src/pia)."""
    p = Path.cwd().resolve()
    if (p / "src" / "pia").is_dir():
        return p
    if (p.parent / "src" / "pia").is_dir():
        return p.parent
    msg = f"No se encontró src/pia. cwd={p}"
    raise FileNotFoundError(msg)


REPO_ROOT = resolve_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_ROOT = str(REPO_ROOT / "data")

# Run IMP (ResNet-18): baseline = fin ronda 0; podado = model_final del mismo run
IMP_RUN_DIR = REPO_ROOT / "runs" / "lt" / "large-checkpoint"
RESNET_BASELINE_CKPT = IMP_RUN_DIR / "round_00" / "model_state.pt"
RESNET_IMP_FINAL_CKPT = IMP_RUN_DIR / "model_final.pt"
IMP_INDEX = IMP_RUN_DIR / "imp_index.json"

# Run poda estructurada (NarrowableChainCnn)
STRUCTURED_RUN_DIR = REPO_ROOT / "runs" / "structured_lt" / "demo"
STRUCTURED_FINAL_CKPT = STRUCTURED_RUN_DIR / "model_final.pt"
STRUCTURED_INDEX = STRUCTURED_RUN_DIR / "structured_index.json"

BATCH_SIZE = 128
MAX_TIMING_BATCHES = 30
SPARSITY_EPS = 1e-6

print("REPO_ROOT:", REPO_ROOT)
print("RESNET_BASELINE_CKPT:", RESNET_BASELINE_CKPT)
print("RESNET_IMP_FINAL_CKPT:", RESNET_IMP_FINAL_CKPT)
print("STRUCTURED_FINAL_CKPT:", STRUCTURED_FINAL_CKPT)

In [ ]:
"""Utilidades: dispositivo, carga, ResNet, cadena estructurada, eval y timing."""

from __future__ import annotations

import json
import math
import statistics
import time
from typing import Any

import torch
from torch import nn

from pia.data.cifar10 import build_cifar10_test_loader
from pia.models.resnet_cifar import apply_he_init, build_resnet18_cifar
from pia.pruning.prune_structured import NarrowableChainCnn
from pia.training.metrics import weight_sparsity_ratio


def pick_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def load_state_dict(path: Path) -> dict[str, torch.Tensor]:
    try:
        raw = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        raw = torch.load(path, map_location="cpu")
    if not isinstance(raw, dict):
        msg = "Se esperaba un dict (state_dict)."
        raise TypeError(msg)
    return raw


def model_memory_bytes(module: nn.Module) -> tuple[int, int, int]:
    param_b = sum(p.numel() * p.element_size() for p in module.parameters())
    buf_b = sum(b.numel() * b.element_size() for b in module.buffers())
    n_params = sum(p.numel() for p in module.parameters())
    return param_b, buf_b, n_params


def build_resnet_from_checkpoint(sd: dict[str, torch.Tensor]) -> nn.Module:
    model = build_resnet18_cifar(num_classes=10)
    apply_he_init(model)
    model.load_state_dict(sd)
    return model


def build_narrowable_chain_from_state_dict(sd: dict[str, torch.Tensor]) -> NarrowableChainCnn:
    """
    Reconstruye ``NarrowableChainCnn`` a partir del ``state_dict`` guardado
    por ``iterative_structured_magnitude_pruning`` (inferencia de H=W desde fc).
    """
    w1 = sd["conv1.weight"]
    width, in_ch, *_ = w1.shape
    fc_w = sd["fc.weight"]
    _n_cls, flat = fc_w.shape
    if flat % width != 0:
        msg = f"fc.in_features={flat} no divisible por width={width}"
        raise ValueError(msg)
    hw = flat // width
    s = int(math.isqrt(hw))
    if s * s != hw:
        msg = f"Mapa no cuadrado: hw={hw}"
        raise ValueError(msg)
    n_cls = int(fc_w.shape[0])
    model = NarrowableChainCnn(in_ch, width, (s, s), n_cls)
    model.load_state_dict(sd)
    return model


def eval_test_accuracy(model: nn.Module, loader: Any, device: torch.device) -> float:
    model.eval()
    model.to(device)
    total_correct = 0
    total_n = 0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            pred = logits.argmax(dim=1)
            total_correct += int((pred == y).sum().item())
            total_n += int(y.shape[0])
    return total_correct / max(1, total_n)


def time_forward_batches(
    model: nn.Module,
    loader: Any,
    device: torch.device,
    *,
    max_batches: int,
) -> tuple[float, float]:
    model.eval()
    model.to(device)
    it = iter(loader)
    x0, _ = next(it)
    x0 = x0.to(device, non_blocking=True)
    with torch.no_grad():
        _ = model(x0)
    if device.type == "cuda":
        torch.cuda.synchronize()
    lat_ms: list[float] = []
    n = 0
    with torch.no_grad():
        for x, _ in loader:
            if n >= max_batches:
                break
            x = x.to(device, non_blocking=True)
            t0 = time.perf_counter()
            _ = model(x)
            if device.type == "cuda":
                torch.cuda.synchronize()
            lat_ms.append((time.perf_counter() - t0) * 1000.0)
            n += 1
    if len(lat_ms) < 2:
        m = float(lat_ms[0]) if lat_ms else 0.0
        return m, 0.0
    return statistics.mean(lat_ms), statistics.stdev(lat_ms)


def disk_mib(path: Path) -> float:
    return path.stat().st_size / (1024.0 * 1024.0)


device = pick_device()
print("device:", device)

In [ ]:
"""Métricas de entrenamiento registradas en los índices JSON (referencia)."""

from typing import Any

import pandas as pd


def _load_rounds_index(path: Path) -> list[dict[str, Any]]:
    if not path.is_file():
        return []
    raw = json.loads(path.read_text(encoding="utf-8"))
    if isinstance(raw, list):
        return [r for r in raw if isinstance(r, dict)]
    rounds = raw.get("rounds")
    if isinstance(rounds, list):
        return [r for r in rounds if isinstance(r, dict)]
    return []


imp_rounds = _load_rounds_index(IMP_INDEX)
struct_rounds = _load_rounds_index(STRUCTURED_INDEX)

rows_imp: list[dict[str, Any]] = []
for r in imp_rounds:
    fm = r.get("final_metrics", {})
    rows_imp.append(
        {
            "source": "imp_resnet",
            "round": r.get("round"),
            "val_acc_logged": fm.get("val/acc"),
            "achieved_sparsity": r.get("achieved_sparsity"),
        }
    )
rows_struct: list[dict[str, Any]] = []
for r in struct_rounds:
    fm = r.get("final_metrics", {})
    rows_struct.append(
        {
            "source": "structured_chain",
            "round": r.get("round"),
            "val_acc_logged": fm.get("val/acc"),
            "struct_width": r.get("structured_conv_width"),
            "achieved_sparsity": r.get("achieved_sparsity"),
        }
    )

if rows_imp:
    display(pd.DataFrame(rows_imp))
else:
    print("(Sin imp_index.json o vacío)")

if rows_struct:
    display(pd.DataFrame(rows_struct))
else:
    print("(Sin structured_index.json o vacío)")

In [ ]:
"""Test CIFAR-10 (misma evaluación para los tres modelos)."""

test_loader = build_cifar10_test_loader(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=0,
)
print("batches test:", len(test_loader))

In [ ]:
"""Tabla comparativa: disco, tensores, params, dispersidad, test acc, latencia."""

import pandas as pd


def row_for_model(
    label: str,
    arch: str,
    ckpt: Path,
    model: nn.Module,
    *,
    extra: dict[str, Any] | None = None,
) -> dict[str, Any]:
    if not ckpt.is_file():
        raise FileNotFoundError(ckpt)
    param_b, buf_b, n_params = model_memory_bytes(model)
    tensores_mib = (param_b + buf_b) / (1024.0 * 1024.0)
    sp = weight_sparsity_ratio(model, eps=SPARSITY_EPS)
    acc = eval_test_accuracy(model, test_loader, device)
    mean_ms, std_ms = time_forward_batches(
        model, test_loader, device, max_batches=MAX_TIMING_BATCHES
    )
    out: dict[str, Any] = {
        "modelo": label,
        "arquitectura": arch,
        "checkpoint": str(ckpt.relative_to(REPO_ROOT)),
        "disco_MiB": disk_mib(ckpt),
        "tensores_MiB": tensores_mib,
        "n_params": n_params,
        "weight_sparsity_ratio_eps": sp,
        "test_acc": acc,
        "latency_mean_ms": mean_ms,
        "latency_std_ms": std_ms,
    }
    if extra:
        out.update(extra)
    return out


rows: list[dict[str, Any]] = []

sd_b = load_state_dict(RESNET_BASELINE_CKPT)
m_b = build_resnet_from_checkpoint(sd_b)
rows.append(
    row_for_model("resnet_baseline_r00", "ResNet-18 CIFAR", RESNET_BASELINE_CKPT, m_b)
)

sd_s = load_state_dict(STRUCTURED_FINAL_CKPT)
m_s = build_narrowable_chain_from_state_dict(sd_s)
meta_w = m_s.width
meta_init = None
if STRUCTURED_INDEX.is_file():
    sj = json.loads(STRUCTURED_INDEX.read_text(encoding="utf-8"))
    meta_init = sj.get("meta", {}).get("initial_conv_width")
width_frac = (
    float(meta_w) / float(meta_init) if meta_init else float("nan")
)
rows.append(
    row_for_model(
        "structured_chain_final",
        "NarrowableChainCnn",
        STRUCTURED_FINAL_CKPT,
        m_s,
        extra={
            "conv_width": meta_w,
            "width_vs_initial": width_frac,
        },
    )
)

sd_p = load_state_dict(RESNET_IMP_FINAL_CKPT)
m_p = build_resnet_from_checkpoint(sd_p)
rows.append(
    row_for_model("resnet_imp_final", "ResNet-18 CIFAR", RESNET_IMP_FINAL_CKPT, m_p)
)

df = pd.DataFrame(rows)
display(df)